# Pandas — мини-проект

Сквозной мини-проект: один датасет, решения передаются между этапами. См. docs/project_notebook.md.


In [ ]:
# Практика к уроку — выполняйте ячейки по порядку
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)



## Постановка задачи `(intro)`


Сквозной пайплайн на **Titanic (OpenML)**: от сырого `df` до таблицы для sklearn — каждый шаг **меняет** `df` и передаёт результат дальше.

**Сюжет:** load → EDA → impute/FE → groupby/merge → OHE → Parquet → split → Pipeline.


## Загрузка `(eda)`


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_openml

%matplotlib inline
np.random.seed(42)
sns.set_theme(style="whitegrid", font_scale=1.05)

raw = fetch_openml("titanic", version=1, as_frame=True, parser="auto")
df = raw.frame.copy()
df["Survived"] = df["survived"].astype(int)
df["Pclass"] = df["pclass"].astype(int)
df["Age"] = pd.to_numeric(df["age"], errors="coerce")
df["Fare"] = pd.to_numeric(df["fare"], errors="coerce")
df["Sex"] = df["sex"].astype(str)
df["Embarked"] = df["embarked"].astype(str)

print(f"Объектов: {len(df)}")
print("Пропуски:\n", df[["Age", "Fare", "Embarked"]].isnull().sum())



## EDA: что в данных? `(eda)`


Видим пропуски в Age/Fare и дисбаланс классов. `name` — почти ID, в модель не берём.


In [ ]:
print(df["Sex"].value_counts())
print("nunique name:", df["name"].nunique())
corr = df[["Survived", "Pclass", "Age", "Fare"]].corr()
sns.heatmap(corr, annot=True, cmap="RdBu_r", center=0, vmin=-1, vmax=1)
plt.title("corr() — отправная точка")
plt.tight_layout()
plt.show()



## Решение: заполнение пропусков `(example)`


**Решение:** Age — медиана по Pclass; Embarked — мода; Fare — медиана по Pclass. Результат — `df_clean`.


In [ ]:
df_clean = df.copy()
df_clean["Age"] = df_clean.groupby("Pclass")["Age"].transform(lambda s: s.fillna(s.median()))
df_clean["Embarked"] = df_clean["Embarked"].fillna(df_clean["Embarked"].mode()[0])
df_clean["Fare"] = df_clean["Fare"].fillna(df_clean.groupby("Pclass")["Fare"].transform("median"))
print("Пропуски после impute:", df_clean[["Age", "Fare", "Embarked"]].isnull().sum().sum())



## Решение: feature engineering `(example)`


**Решение:** clip Fare на 99-м перцентиле, бинарный `is_expensive`, отклонение от среднего fare по классу (`transform`).


In [ ]:
upper = df_clean["Fare"].quantile(0.99)
df_clean["Fare_clip"] = df_clean["Fare"].clip(upper=upper)
df_clean["is_expensive"] = np.where(df_clean["Fare"] > df_clean["Fare"].median(), 1, 0)
df_clean["fare_dev"] = df_clean["Fare"] - df_clean.groupby("Pclass")["Fare"].transform("mean")
df_clean[["Fare", "Fare_clip", "fare_dev", "is_expensive"]].head()



## groupby и merge `(viz)`


Агрегат по Pclass×Sex и join справочника портов — всё на **df_clean**.


In [ ]:
summary = df_clean.groupby(["Pclass", "Sex"]).agg(
    n=("Survived", "size"),
    survival=("Survived", "mean"),
    avg_fare=("Fare", "mean"),
).round(3)
print(summary)

ports = pd.DataFrame({
    "Embarked": ["S", "C", "Q"],
    "port_name": ["Southampton", "Cherbourg", "Queenstown"],
})
df_clean = pd.merge(df_clean, ports, on="Embarked", how="left")
df_clean.groupby("port_name")["Survived"].mean().sort_values(ascending=False)



## Решение: таблица для модели `(example)`


**Решение:** OHE для Sex/Embarked, отбираем числовые признаки → `df_model`. Сохраняем в Parquet.


In [ ]:
df_model = pd.get_dummies(df_clean, columns=["Sex", "Embarked"], drop_first=True)
feature_cols = ["Pclass", "Age", "Fare_clip", "is_expensive", "fare_dev"] + [
    c for c in df_model.columns if c.startswith("Sex_") or c.startswith("Embarked_")
]
df_model = df_model[feature_cols + ["Survived"]].reset_index(drop=True)

import tempfile
from pathlib import Path
pq_path = Path(tempfile.mkdtemp()) / "titanic_clean.parquet"
try:
    df_model.to_parquet(pq_path, index=False)
    df_model = pd.read_parquet(pq_path)
    print("Parquet:", df_model.shape)
except ImportError:
    csv_path = pq_path.with_suffix(".csv")
    df_model.to_csv(csv_path, index=False)
    df_model = pd.read_csv(csv_path)
    print("CSV (pyarrow нет):", df_model.shape)



## Решение: split и Pipeline `(example)`


**Решение:** split **после** всех трансформаций; imputer/scaler — только в Pipeline на train.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

X = df_model.drop(columns=["Survived"])
y = df_model["Survived"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

final_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=500, random_state=42)),
])
final_pipe.fit(X_train, y_train)
pred = final_pipe.predict(X_test)
print(classification_report(y_test, pred, digits=3))



## Чек-лист `(summary)`


1. Один `df` → `df_clean` → `df_model` — без перезагрузки.
2. Каждый шаг явно **решает** проблему предыдущего.
3. `transform` для контекстных признаков; merge по уникальному ключу.
4. OHE + Parquet; split **после** FE.
5. Pipeline — imputer/scaler без утечки из test.
